# 07 — Mask-aware validation (P0, project decision 2026-09-04)

An Opus-model review of notebook 06's results found a plausible mechanism for why
the internal proxy (`evaluate.horizon_matched_split`) got each feature's individual
sign right (climatology helps, trend hurts) but completely inverted the
climatology+trend *interaction* ranking on the real leaderboard: **66.5% of
Test.csv's `TWS_t` is masked-then-backward-filled - frozen at its last observed
value - while `horizon_matched_split` validates on fully-observed `TWS_t`,
every time.** A rolling trend slope computed over a frozen/repeated tail collapses
toward a degenerate value at real inference, but looked informative in validation
where it saw genuinely varying data. Climatology (cross-year, same-calendar-month)
is barely touched by this, since masking only affects the most recent 1-6 months,
not prior years.

This notebook builds a validation split that simulates Test.csv's real masking
pattern on the validation fold *before* computing derived features - so validation
faces the same frozen-tail regime as real inference - and checks whether it
reproduces the real leaderboard's ranking across the four configurations we now
have real scores for:

| Config | Real Zindi RMSE |
|---|---|
| baseline (no climatology, no trend) | 0.7822 |
| + climatology | **0.7778** (best) |
| + trend | 0.7834 |
| + climatology + trend | 0.7845 (worst) |

`evaluate.measure_masking_pattern`, `evaluate.simulate_masking`,
`evaluate.mask_aware_horizon_matched_split`, and `features.build_all_features`
(extracted from `src/train.py` so validation and production run the identical
feature pipeline) are already implemented and unit-tested in `src/` - this
notebook is their validation, per this project's graduation rule.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import config, data, evaluate, features, model

train, test, sample_submission = data.load_raw_data()
print("train:", train.shape, "| test:", test.shape)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



train: (2154021, 13) | test: (280961, 13)


## Measure Test.csv's real masking pattern

`evaluate.measure_masking_pattern` replaces notebook 03's hardcoded guess
(12/18 months, 99.5% row fraction) with numbers measured directly from Test.csv.

In [2]:
masked_month_fraction, masked_row_fraction = evaluate.measure_masking_pattern(test)
print(f"Masked-month fraction: {masked_month_fraction:.3f} (nb01 estimate: {12/18:.3f})")
print(f"Row fraction within masked months: {masked_row_fraction:.3f} (nb01 estimate: ~0.995)")

target_horizons = evaluate.compute_test_horizons(train, test)
print("Target horizons:", sorted(target_horizons))

Masked-month fraction: 0.667 (nb01 estimate: 0.667)
Row fraction within masked months: 0.998 (nb01 estimate: ~0.995)
Target horizons: [1, 5, 6, 7, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 35, 39, 40]


## Build the mask-aware fit/validation split once

`mask_aware_horizon_matched_split` takes RAW train (masking must happen before
derived features are computed, exactly mirroring Test.csv's own pipeline order),
and returns fully feature-engineered fit/val frames (fill + neighbourhood +
climatology already applied, via `features.build_all_features`).

In [3]:
fit_df, val_df = evaluate.mask_aware_horizon_matched_split(
    train, target_horizons, masked_month_fraction, masked_row_fraction,
)
print(f"Fit: {len(fit_df)} rows | Val: {len(val_df)} rows")
print(f"Val TWS_t masked share (post-simulation, pre-fill): "
      f"n/a (already filled by build_all_features) - "
      f"val rows still missing after fill: {val_df[config.TWS_COL].isna().sum()}")

Fit: 1716994 rows | Val: 187342 rows
Val TWS_t masked share (post-simulation, pre-fill): n/a (already filled by build_all_features) - val rows still missing after fill: 0


## Add the long-window trend feature on top

Not part of `build_all_features` (never graduated - confirmed negative result).
Reusing notebook 06's implementation to test the `+trend` and `+climatology+trend`
configurations under this new proxy too.

In [4]:
def add_long_window_trend_feature(df: pd.DataFrame, window_size: int = 24) -> pd.DataFrame:
    out = df.copy()
    ordered = out.sort_values([config.LAT_COL, config.LON_COL, config.TIME_COL]).copy()
    ordered["_month_num"] = ordered[config.TIME_COL].dt.year * 12 + ordered[config.TIME_COL].dt.month

    x = ordered["_month_num"].astype(float)
    y = ordered[config.TWS_COL]
    group_keys = [ordered[config.LAT_COL], ordered[config.LON_COL]]
    min_periods = max(3, window_size // 4)

    def roll_mean(s):
        return s.groupby(group_keys).transform(
            lambda v: v.rolling(window_size, min_periods=min_periods).mean()
        )

    mean_x, mean_y = roll_mean(x), roll_mean(y)
    mean_xy, mean_x2 = roll_mean(x * y), roll_mean(x * x)

    var_x = mean_x2 - mean_x ** 2
    cov_xy = mean_xy - mean_x * mean_y
    with np.errstate(invalid="ignore", divide="ignore"):
        slope = (cov_xy / var_x).replace([np.inf, -np.inf], np.nan)

    out["tws_trend_slope"] = slope.reindex(out.index)
    return out


n_fit = len(fit_df)
combined_trend = add_long_window_trend_feature(pd.concat([fit_df, val_df], ignore_index=True))
fit_trend = combined_trend.iloc[:n_fit].reset_index(drop=True)
val_trend = combined_trend.iloc[n_fit:].reset_index(drop=True)
print(f"Trend coverage (val): {val_trend['tws_trend_slope'].notna().mean():.1%}")

Trend coverage (val): 100.0%


## Score all four configurations under the new mask-aware proxy

In [5]:
def score(fit, val, cols):
    X_fit = features.select_base_features(fit, cols)
    y_fit = fit[config.TARGET_COL].to_numpy()
    X_val = features.select_base_features(val, cols)
    y_val = val[config.TARGET_COL].to_numpy()
    m = model.make_baseline_model()
    m.fit(X_fit, y_fit)
    y_pred = model.predict(m, X_val)
    return evaluate.compute_metrics(y_val, y_pred)


base_cols = ["TWS_t", "month_sin", "month_cos", "SPEI_01_t", "SPEI_03_t", "SPEI_06_t",
             "SPEI_12_t", "SOIL_MOISTURE_t"] + config.NEIGHBOURHOOD_FEATURE_COLS
clim_cols = base_cols + config.CLIMATOLOGY_FEATURE_COLS
trend_cols = base_cols + ["tws_trend_slope"]
both_cols = clim_cols + ["tws_trend_slope"]

results = {
    "baseline": score(fit_df, val_df, base_cols),
    "+ climatology": score(fit_df, val_df, clim_cols),
    "+ trend": score(fit_trend, val_trend, trend_cols),
    "+ climatology + trend": score(fit_trend, val_trend, both_cols),
}
new_proxy = pd.DataFrame(results).T
new_proxy["real_zindi_rmse"] = [0.7822, 0.7778, 0.7834, 0.7845]
new_proxy["old_proxy_rmse"] = [0.6532, 0.6505, 0.653493, 0.646349]
new_proxy

C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _wina

,rmse,mae,r2,real_zindi_rmse,old_proxy_rmse
baseline,0.755183,0.540749,0.231845,0.7822,0.653200
+ climatology,0.746920,0.544200,0.248562,0.7778,0.650500
+ trend,0.754918,0.541119,0.232384,0.7834,0.653493
+ climatology + trend,0.750006,0.547272,0.242339,0.7845,0.646349


## Does the new proxy's ranking match reality?

In [6]:
ranking_new = new_proxy["rmse"].rank()
ranking_real = new_proxy["real_zindi_rmse"].rank()
ranking_old = new_proxy["old_proxy_rmse"].rank()

print("Ranking (1=best) - new mask-aware proxy vs. real vs. old proxy:")
print(pd.DataFrame({
    "new_proxy_rank": ranking_new, "real_rank": ranking_real, "old_proxy_rank": ranking_old,
}))
print()
print("New proxy matches real ranking exactly:", ranking_new.equals(ranking_real))
print("Old proxy matched real ranking exactly:", ranking_old.equals(ranking_real))

Ranking (1=best) - new mask-aware proxy vs. real vs. old proxy:
                       new_proxy_rank  real_rank  old_proxy_rank
baseline                          4.0        2.0             3.0
+ climatology                     1.0        1.0             2.0
+ trend                           3.0        3.0             4.0
+ climatology + trend             2.0        4.0             1.0

New proxy matches real ranking exactly: False
Old proxy matched real ranking exactly: False


## Gate decision

**Partial win - graduate the split, but the interaction mystery is not closed.**

**Big win: absolute calibration.** The new proxy (0.747-0.755 RMSE across the four
configs) sits far closer to the real leaderboard (0.778-0.785) than the old
horizon-matched-only proxy did (0.646-0.653). Gap-closed on the baseline config:
(0.7822 - 0.6532) = 0.129 old gap -> (0.7822 - 0.755183) = 0.027 new gap, a **79%
reduction** - matching the Opus review's back-of-envelope estimate almost exactly.
This alone is worth graduating: every future feature-gate decision will be made
against a much more honest number.

**Not a full win: the interaction ranking is still wrong.** Real order (ascending
RMSE): climatology < baseline < trend < combined. New-proxy order: climatology <
combined < trend < baseline. Climatology-is-best is now correctly captured - but
the new proxy still ranks climatology+trend combined as 2nd-best, when in reality
it is the single worst option, worse than doing nothing. The masking-skew
hypothesis (frozen recent `TWS_t` values making a trend slope degenerate at
inference) explains most of the *absolute* gap but does not, by itself, explain
why *combining* trend with climatology specifically backfires in the real world.
Simulating masking within Train.csv's 2002-2015 window still cannot reproduce a
genuine out-of-time effect the way real 2016-2018 Test.csv data would - so nb05's
distribution-shift hypothesis is not ruled out for this specific interaction,
even though the Opus review's quick check of observed-month statistics did not
support it as the *dominant* explanation for the wider gap.

**Decision:** graduate `evaluate.mask_aware_horizon_matched_split` (+
`measure_masking_pattern`, `simulate_masking`, `features.build_all_features`) as
this project's new primary internal proxy - already implemented and unit-tested
in `src/`, `src/train.py` updated to report it. **Keep the Tier A/Tier B gate
policy from the Opus review regardless of this only-partial fix**: features whose
mechanism is backward-looking/seasonal/spatial (climatology-like) can be gated on
this proxy; anything that reads or extrapolates the recent `TWS_t` trajectory
(trend-like) still needs a real Zindi submission before graduating, since even
this improved proxy did not reliably predict trend-combination behaviour. Trend
itself remains a confirmed negative result, not graduated.
